# Pipeline de Vendas de Supermercado — PostgreSQL + Python

Notebook com o fluxo completo do projeto, passo a passo. O código de cada
célula é o mesmo dos scripts em `src/`, que automatizam o pipeline
(`python src/run_etl.py`).

**Fonte dos dados:** [Supermarket Sales Dataset (Kaggle)](https://www.kaggle.com/datasets/faresashraf1001/supermarket-sales)

| Fase | O que faz |
|---|---|
| 0 | Cria o banco e as tabelas (`sql/01` e `sql/02`) |
| 1 | Carga Raw: CSV → `raw_vendas`, sem alterar o conteúdo |
| 2 | Consultas SQL (`sql/03`) e exportação para CSV |
| 2/3 | Leitura e inspeção inicial no Pandas |
| 3 | ETL: tipagem, nulos, duplicidades, colunas derivadas → `vendas_tratadas` |
| 4 | Estatística, hipóteses, respostas de negócio e gráficos |

**Antes de executar:** `pip install -r requirements.txt`, criar o `.env`
(copie de `.env.example`) e salvar o CSV do Kaggle em
`data/raw/SuperMarket Analysis.csv`. Execute este notebook a partir da raiz
do projeto (ou da pasta onde ele estiver salvo).

## Configuração
Caminhos, credenciais (`.env`) e engines de conexão.

In [ ]:
# Configuracoes centrais: caminhos, credenciais e engines.
import os
from pathlib import Path

from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Raiz do projeto: pasta que contém sql/ e src/
BASE_DIR = Path.cwd() if (Path.cwd() / "sql").exists() else Path.cwd().parent
load_dotenv(BASE_DIR / ".env")

# ---------- Banco de dados ----------
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = int(os.getenv("DB_PORT", "5432"))
DB_ADMIN = os.getenv("DB_ADMIN", "postgres")
DB_NAME = os.getenv("DB_NAME", "supermercado")

# ---------- Caminhos ----------
SQL_DIR = BASE_DIR / "sql"
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RESULTADOS_DIR = BASE_DIR / "resultados"
GRAFICOS_DIR = RESULTADOS_DIR / "graficos"

CSV_ORIGEM = RAW_DIR / os.getenv("CSV_NOME", "SuperMarket Analysis.csv")
CSV_EXPORTADO = RAW_DIR / "export_raw_vendas.csv"
CSV_TRATADO = PROCESSED_DIR / "vendas_tratadas.csv"

# Indice = datetime.weekday() (segunda = 0)
DIAS_SEMANA = [
    "Segunda-feira",
    "Terça-feira",
    "Quarta-feira",
    "Quinta-feira",
    "Sexta-feira",
    "Sábado",
    "Domingo",
]


def _engine(database: str):
    url = URL.create(
        drivername="postgresql+psycopg2",
        username=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT,
        database=database,
    )
    return create_engine(url)


def get_engine_admin():
    """Engine do banco padrao (usada so para criar o banco)."""
    return _engine(DB_ADMIN)


def get_engine():
    """Engine do banco do projeto."""
    return _engine(DB_NAME)


def banner(msg: str) -> None:
    """Imprime uma mensagem delimitada por linhas."""
    largura = max(len(linha) for linha in msg.split("\n"))
    barra = "=" * largura
    print(f"{barra}\n{msg}\n{barra}")


## Fase 0 — Banco e tabelas
Executa `sql/01_criar_banco.sql` e `sql/02_criar_tabelas.sql` (chaves primárias, `NOT NULL` e `CHECK`).

In [ ]:
# Fase 0: cria o banco e as tabelas executando os scripts SQL.
from sqlalchemy import text


def ler_sql(nome_arquivo: str) -> str:
    return (SQL_DIR / nome_arquivo).read_text(encoding="utf-8")


def criar_banco() -> None:
    """Executa 01_criar_banco.sql se o banco ainda nao existir."""
    engine = get_engine_admin()
    with engine.connect().execution_options(
        isolation_level="AUTOCOMMIT"
    ) as conn:
        existe = conn.execute(
            text("SELECT 1 FROM pg_database WHERE datname = :nome"),
            {"nome": DB_NAME},
        ).scalar()
        if existe:
            print(f"Banco '{DB_NAME}' ja existe. Nada a fazer.")
        else:
            conn.exec_driver_sql(ler_sql("01_criar_banco.sql"))
            print(f"Banco '{DB_NAME}' criado com sucesso.")
    engine.dispose()


def criar_tabelas() -> None:
    """Executa 02_criar_tabelas.sql (idempotente)."""
    engine = get_engine()
    with engine.begin() as conn:
        conn.exec_driver_sql(ler_sql("02_criar_tabelas.sql"))
    engine.dispose()
    print("Tabelas raw_vendas e vendas_tratadas verificadas/criadas.")


def executar() -> None:
    criar_banco()
    criar_tabelas()
    banner("Fase 0 concluida: banco e tabelas prontos.")


In [ ]:
executar()

## Fase 1 — Carga Raw
Carrega o CSV original em `raw_vendas` com todos os campos em texto, preservando o conteúdo. `TRUNCATE` + `INSERT` na mesma transação.

In [ ]:
# Fase 1: carrega o CSV original, sem alterar o conteudo, em raw_vendas.
import pandas as pd
from sqlalchemy import text


TABELA_RAW = "raw_vendas"

# Nome original no CSV -> coluna da tabela raw (so troca o formato do
# nome; os valores permanecem exatamente como no arquivo).
MAPA_COLUNAS_RAW = {
    "Invoice ID": "invoice_id",
    "Branch": "branch",
    "City": "city",
    "Customer type": "customer_type",
    "Gender": "gender",
    "Product line": "product_line",
    "Unit price": "unit_price",
    "Quantity": "quantity",
    "Tax 5%": "tax_5pct",
    "Sales": "sales",
    "Date": "date",
    "Time": "time",
    "Payment": "payment",
    "cogs": "cogs",
    "gross margin percentage": "gross_margin_percentage",
    "gross income": "gross_income",
    "Rating": "rating",
}


def ler_csv_original() -> pd.DataFrame:
    if not CSV_ORIGEM.exists():
        raise FileNotFoundError(
            f"CSV nao encontrado: {CSV_ORIGEM}\n"
            "Baixe o dataset do Kaggle e salve em data/raw/."
        )
    # dtype=str preserva exatamente o texto de cada celula
    df = pd.read_csv(CSV_ORIGEM, dtype=str)
    faltantes = set(MAPA_COLUNAS_RAW) - set(df.columns)
    if faltantes:
        raise ValueError(f"Colunas ausentes no CSV: {sorted(faltantes)}")
    return df[list(MAPA_COLUNAS_RAW)].rename(columns=MAPA_COLUNAS_RAW)


def executar() -> None:
    df = ler_csv_original()
    engine = get_engine()

    # TRUNCATE + INSERT na mesma transacao: ou carrega tudo ou nada
    with engine.begin() as conn:
        conn.execute(text(f"TRUNCATE TABLE {TABELA_RAW}"))
        df.to_sql(
            TABELA_RAW,
            con=conn,
            if_exists="append",
            index=False,
            method="multi",
            chunksize=500,
        )

    with engine.connect() as conn:
        total_banco = conn.execute(
            text(f"SELECT COUNT(*) FROM {TABELA_RAW}")
        ).scalar()
    engine.dispose()

    banner(
        "Fase 1 concluida: carga Raw.\n"
        f"Linhas no CSV: {len(df)} | Linhas em {TABELA_RAW}: {total_banco}"
    )
    if total_banco != len(df):
        raise RuntimeError("Contagem do banco difere da do CSV!")


In [ ]:
executar()

## Fase 2 — Consultas SQL e exportação para CSV
Cada consulta de `sql/03_consultas.sql` tem um marcador `-- @export arquivo.csv`; os resultados são gravados em `data/raw/`.

In [ ]:
# Fase 2: executa as consultas de sql/03_consultas.sql e exporta CSVs.
import re

import pandas as pd
from sqlalchemy import text


MARCADOR = re.compile(r"^--\s*@export\s+(\S+\.csv)\s*$", re.MULTILINE)


def extrair_consultas(conteudo: str) -> list[tuple[str, str]]:
    """Retorna [(arquivo_csv, sql)] a partir dos marcadores '-- @export'."""
    partes = MARCADOR.split(conteudo)
    # partes = [preambulo, arq1, sql1, arq2, sql2, ...]
    return [
        (partes[i], partes[i + 1].strip())
        for i in range(1, len(partes), 2)
    ]


def executar() -> None:
    conteudo = (SQL_DIR / "03_consultas.sql").read_text(encoding="utf-8")
    consultas = extrair_consultas(conteudo)
    RAW_DIR.mkdir(parents=True, exist_ok=True)

    engine = get_engine()
    with engine.connect() as conn:
        for arquivo, sql in consultas:
            df = pd.read_sql(text(sql), con=conn)
            destino = RAW_DIR / arquivo
            df.to_csv(destino, index=False, encoding="utf-8")
            print(f"[OK] {arquivo}: {len(df)} linhas -> {destino}")
    engine.dispose()

    banner(f"Fase 2 concluida: {len(consultas)} consultas exportadas.")


In [ ]:
executar()

In [ ]:
pd.read_csv(RAW_DIR / "consulta_faturamento_por_filial.csv")

In [ ]:
pd.read_csv(RAW_DIR / "consulta_avaliacao_por_linha_produto.csv")

## Leitura e inspeção inicial no Pandas
Lê o CSV exportado do PostgreSQL e verifica tipos, nulos, duplicidades, categorias, estatísticas e inconsistências.

In [ ]:
# Fase 2/3: leitura do CSV exportado e inspecao inicial (EDA) no Pandas.
import io

import pandas as pd


COLUNAS_CATEGORICAS = [
    "branch",
    "city",
    "customer_type",
    "gender",
    "product_line",
    "payment",
]
COLUNAS_NUMERICAS = [
    "unit_price",
    "quantity",
    "tax_5pct",
    "sales",
    "cogs",
    "gross_income",
    "rating",
]


def inspecionar(df: pd.DataFrame) -> list[str]:
    """Gera o relatorio de inspecao como lista de linhas de texto."""
    out: list[str] = []
    add = out.append

    add(f"Dimensoes: {df.shape[0]} linhas x {df.shape[1]} colunas")

    add("\n--- 1. Tipos de dados (inferidos pelo Pandas) ---")
    buf = io.StringIO()
    df.info(buf=buf)
    add(buf.getvalue())

    add("--- 2. Valores nulos ---")
    nulos = df.isnull().sum()
    add(
        nulos[nulos > 0].to_string()
        if nulos.sum() > 0
        else "Nenhum valor nulo encontrado."
    )

    add("\n--- 3. Duplicidades ---")
    add(f"Linhas totalmente duplicadas: {df.duplicated().sum()}")
    add(f"Invoice IDs repetidos: {df['invoice_id'].duplicated().sum()}")

    add("\n--- 4. Cardinalidade das categorias ---")
    for col in COLUNAS_CATEGORICAS:
        add(f"{col}: {sorted(df[col].dropna().unique().tolist())}")

    add("\n--- 5. Estatisticas descritivas (numericas) ---")
    add(df[COLUNAS_NUMERICAS].describe().round(2).to_string())

    add("\n--- 6. Inconsistencias ---")
    add(f"quantity <= 0: {(df['quantity'] <= 0).sum()}")
    add(f"unit_price < 0: {(df['unit_price'] < 0).sum()}")
    add(f"rating fora de 0-10: {(~df['rating'].between(0, 10)).sum()}")
    dif = (df["cogs"] + df["tax_5pct"] - df["sales"]).abs()
    add(f"sales != cogs + tax (tolerancia 0,01): {(dif > 0.01).sum()}")

    add("\n--- 7. Amostra ---")
    add(df.head(3).to_string())
    return out


def executar() -> pd.DataFrame:
    if not CSV_EXPORTADO.exists():
        raise FileNotFoundError(
            f"{CSV_EXPORTADO} nao existe. Execute antes src/exportacao.py."
        )
    df = pd.read_csv(CSV_EXPORTADO)

    banner(f"Inspecao inicial: {CSV_EXPORTADO.name}")
    linhas = inspecionar(df)
    texto = "\n".join(linhas)
    print(texto)

    RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
    (RESULTADOS_DIR / "inspecao_inicial.txt").write_text(
        texto, encoding="utf-8"
    )
    banner("Inspecao concluida (relatorio em resultados/inspecao_inicial.txt)")
    return df


In [ ]:
df_raw = executar()

## Fase 3 — ETL: camada tratada
Dicionário de dados e funções de transformação. Cada etapa é executada separadamente nas células seguintes.

In [ ]:
# Dicionario de dados da camada tratada (gera CSV e Markdown).
import pandas as pd

COLUNAS = ["coluna", "tipo", "restricoes", "descricao", "origem"]

DICIONARIO = [
    ("id_venda", "VARCHAR(50)", "PRIMARY KEY, NOT NULL",
     "Identificador unico da venda (Invoice ID)", "raw: invoice_id"),
    ("Filial", "VARCHAR(10)", "NOT NULL",
     "Codigo da filial", "raw: branch"),
    ("Cidade", "VARCHAR(100)", "NOT NULL",
     "Cidade da filial", "raw: city"),
    ("tipo_cliente", "VARCHAR(50)", "-",
     "Member ou Normal", "raw: customer_type"),
    ("Gênero", "VARCHAR(20)", "-",
     "Genero do cliente", "raw: gender"),
    ("linha_produto", "VARCHAR(150)", "NOT NULL",
     "Categoria do produto", "raw: product_line"),
    ("preco_unitario", "NUMERIC(10,2)", "NOT NULL, CHECK >= 0",
     "Preco por unidade", "raw: unit_price"),
    ("Quantidade", "INTEGER", "NOT NULL, CHECK > 0",
     "Unidades vendidas", "raw: quantity"),
    ("Imposto", "NUMERIC(10,2)", "CHECK >= 0",
     "Imposto de 5 por cento sobre o custo", "raw: tax_5pct"),
    ("valor_total", "NUMERIC(12,2)", "NOT NULL, CHECK >= 0",
     "Valor total da venda (custo + imposto)", "raw: sales"),
    ("data_venda", "DATE", "NOT NULL",
     "Data da venda (texto m/d/Y convertido para data)", "raw: date"),
    ("hora_venda", "TIME", "-",
     "Horario da venda (texto convertido para hora)", "raw: time"),
    ("forma_pagamento", "VARCHAR(50)", "NOT NULL",
     "Cash, Ewallet ou Credit card", "raw: payment"),
    ("custo_mercadoria", "NUMERIC(12,2)", "CHECK >= 0",
     "Custo da mercadoria vendida (cogs)", "raw: cogs"),
    ("margem_percentual", "NUMERIC(10,2)", "-",
     "Margem bruta em percentual", "raw: gross_margin_percentage"),
    ("receita_bruta", "NUMERIC(12,2)", "CHECK >= 0",
     "Receita bruta (gross income)", "raw: gross_income"),
    ("Avaliação", "NUMERIC(4,2)", "CHECK entre 0 e 10",
     "Nota do cliente", "raw: rating"),
    ("dia_semana", "VARCHAR(15)", "NOT NULL",
     "Dia da semana em portugues", "derivada de data_venda"),
    ("mes", "INTEGER", "CHECK entre 1 e 12",
     "Mes da venda", "derivada de data_venda"),
    ("periodo_dia", "VARCHAR(15)", "-",
     "Manha (ate 11h), Tarde (12h-17h), Noite (18h+)",
     "derivada de hora_venda"),
    ("valor_conferido", "BOOLEAN", "-",
     "True se Quantidade x preco_unitario + Imposto = valor_total "
     "(tolerancia 0,02)", "derivada (conferencia)"),
]


def gerar_dicionario(pasta) -> None:
    """Grava dicionario_dados.csv e dicionario_dados.md em `pasta`."""
    df = pd.DataFrame(DICIONARIO, columns=COLUNAS)
    df.to_csv(pasta / "dicionario_dados.csv", index=False, encoding="utf-8")

    linhas = ["# Dicionario de dados - vendas_tratadas", ""]
    linhas.append("| " + " | ".join(COLUNAS) + " |")
    linhas.append("|" + "---|" * len(COLUNAS))
    for registro in DICIONARIO:
        linhas.append("| " + " | ".join(registro) + " |")
    (pasta / "dicionario_dados.md").write_text(
        "\n".join(linhas) + "\n", encoding="utf-8"
    )


In [ ]:
# Fase 3: limpeza, tipagem, nulos, duplicidades e colunas derivadas.
import numpy as np
import pandas as pd
from sqlalchemy import text


# raw (snake_case) -> nome do dicionario de dados
MAPA_COLUNAS = {
    "invoice_id": "id_venda",
    "branch": "Filial",
    "city": "Cidade",
    "customer_type": "tipo_cliente",
    "gender": "Gênero",
    "product_line": "linha_produto",
    "unit_price": "preco_unitario",
    "quantity": "Quantidade",
    "tax_5pct": "Imposto",
    "sales": "valor_total",
    "date": "data_venda",
    "time": "hora_venda",
    "payment": "forma_pagamento",
    "cogs": "custo_mercadoria",
    "gross_margin_percentage": "margem_percentual",
    "gross_income": "receita_bruta",
    "rating": "Avaliação",
}
TEXTO = [
    "id_venda", "Filial", "Cidade", "tipo_cliente", "Gênero",
    "linha_produto", "forma_pagamento", "data_venda", "hora_venda",
]
NUMERICAS = [
    "preco_unitario", "Imposto", "valor_total", "custo_mercadoria",
    "margem_percentual", "receita_bruta", "Avaliação",
]
OBRIGATORIAS = [
    "id_venda", "Filial", "Cidade", "linha_produto", "preco_unitario",
    "Quantidade", "valor_total", "data_venda", "forma_pagamento",
]
OPCIONAIS_TEXTO = ["tipo_cliente", "Gênero"]
DERIVADAS = ["dia_semana", "mes", "periodo_dia", "valor_conferido"]
COLUNAS_FINAIS = list(MAPA_COLUNAS.values()) + DERIVADAS
TABELA_TRATADA = "vendas_tratadas"


# ---------------------------------------------------------------- extrair
def extrair() -> pd.DataFrame:
    """Le a camada Raw (tudo texto) do PostgreSQL."""
    colunas = ", ".join(MAPA_COLUNAS)
    engine = get_engine()
    with engine.connect() as conn:
        df = pd.read_sql(
            text(f"SELECT {colunas} FROM raw_vendas ORDER BY invoice_id"),
            con=conn,
        )
    engine.dispose()
    print(f"Registros lidos de raw_vendas: {len(df)}")
    return df.rename(columns=MAPA_COLUNAS)


# ------------------------------------------------------------------ limpar
def limpar_texto(df: pd.DataFrame) -> pd.DataFrame:
    """Remove espacos extras e transforma vazios em nulo."""
    for col in TEXTO:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .replace("", pd.NA)
        )
    return df


def _parse_datetime(serie: pd.Series, formato: str) -> pd.Series:
    """Tenta o formato esperado e cai para inferencia por elemento."""
    convertido = pd.to_datetime(serie, format=formato, errors="coerce")
    if convertido.isna().any():
        alternativa = pd.to_datetime(serie, format="mixed", errors="coerce")
        convertido = convertido.fillna(alternativa)
    return convertido


def converter_tipos(df: pd.DataFrame) -> pd.DataFrame:
    """Casting: numericos, inteiro, data e hora."""
    print("\n--- Conversao de tipos ---")
    for col in NUMERICAS + ["Quantidade"]:
        antes = df[col].notna().sum()
        df[col] = pd.to_numeric(df[col], errors="coerce")
        perdidos = antes - df[col].notna().sum()
        if perdidos:
            print(f"{col}: {perdidos} valor(es) invalido(s) -> nulo")

    # Quantidade precisa ser inteira: fracionados viram nulo
    fracionados = df["Quantidade"].notna() & (df["Quantidade"] % 1 != 0)
    if fracionados.any():
        print(f"Quantidade fracionada -> nulo: {fracionados.sum()}")
        df.loc[fracionados, "Quantidade"] = np.nan
    df["Quantidade"] = df["Quantidade"].astype("Int64")

    df["data_venda"] = _parse_datetime(df["data_venda"], "%m/%d/%Y")
    df["hora_venda"] = _parse_datetime(df["hora_venda"], "%H:%M")
    print("Tipos convertidos com sucesso.")
    return df


# ------------------------------------------------------------------- nulos
def tratar_nulos(df: pd.DataFrame) -> pd.DataFrame:
    print("\n--- Valores ausentes ---")
    nulos = df.isna().sum()
    print(
        nulos[nulos > 0].to_string()
        if nulos.sum() > 0
        else "Nenhum valor ausente."
    )

    for col in OPCIONAIS_TEXTO:
        df[col] = df[col].fillna("Não informado")

    antes = len(df)
    df = df.dropna(subset=OBRIGATORIAS)
    print(f"Linhas removidas por nulo em campo obrigatorio: "
          f"{antes - len(df)}")
    return df


# ------------------------------------------------------------- duplicidade
def remover_duplicidades(df: pd.DataFrame) -> pd.DataFrame:
    print("\n--- Duplicidades ---")
    antes = len(df)
    df = df.drop_duplicates()
    print(f"Linhas totalmente duplicadas removidas: {antes - len(df)}")

    antes = len(df)
    df = df.drop_duplicates(subset="id_venda", keep="first")
    print(f"IDs de venda repetidos removidos: {antes - len(df)}")
    return df


# ------------------------------------------------------------------ regras
def validar_regras(df: pd.DataFrame) -> pd.DataFrame:
    """Remove registros que violariam os CHECKs do banco."""
    print("\n--- Regras de negocio (casos limitrofes) ---")

    def valido(col, cond):
        return cond(df[col]) | df[col].isna()

    regras = {
        "preco_unitario >= 0": valido("preco_unitario", lambda s: s >= 0),
        "Quantidade > 0": valido("Quantidade", lambda s: s > 0),
        "Imposto >= 0": valido("Imposto", lambda s: s >= 0),
        "valor_total >= 0": valido("valor_total", lambda s: s >= 0),
        "custo_mercadoria >= 0": valido(
            "custo_mercadoria", lambda s: s >= 0),
        "receita_bruta >= 0": valido("receita_bruta", lambda s: s >= 0),
        "Avaliação entre 0 e 10": valido(
            "Avaliação", lambda s: s.between(0, 10)),
    }
    mascara = pd.Series(True, index=df.index)
    for nome, ok in regras.items():
        invalidos = (~ok).sum()
        print(f"{nome}: {invalidos} violacao(oes)")
        mascara &= ok
    print(f"Linhas removidas por regras: {(~mascara).sum()}")
    return df[mascara].copy()


# ---------------------------------------------------------------- derivadas
def criar_derivadas(df: pd.DataFrame) -> pd.DataFrame:
    """Feature engineering: dia_semana, mes, periodo_dia, valor_conferido."""
    df["dia_semana"] = df["data_venda"].dt.weekday.map(
        lambda i: DIAS_SEMANA[i]
    )
    df["mes"] = df["data_venda"].dt.month

    hora = df["hora_venda"].dt.hour
    df["periodo_dia"] = np.select(
        [hora.isna(), hora < 12, hora < 18],
        ["Não informado", "Manhã", "Tarde"],
        default="Noite",
    )

    esperado = (
        df["Quantidade"].astype("float64") * df["preco_unitario"]
        + df["Imposto"]
    )
    df["valor_conferido"] = ((esperado - df["valor_total"]).abs() <= 0.02)
    nao_conf = (~df["valor_conferido"]).sum()
    print(f"\nVendas com valor_total divergente do calculado: {nao_conf}")
    return df


# ------------------------------------------------------------------ salvar
def finalizar(df: pd.DataFrame) -> pd.DataFrame:
    """Arredonda para 2 casas (NUMERIC x,2) e converte data/hora."""
    for col in NUMERICAS:
        df[col] = df[col].round(2)
    df["data_venda"] = df["data_venda"].dt.date
    df["hora_venda"] = df["hora_venda"].dt.time
    for col in df.select_dtypes("string").columns:
        df[col] = df[col].astype(object)
    return df[COLUNAS_FINAIS].reset_index(drop=True)


def salvar_csv(df: pd.DataFrame) -> None:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(CSV_TRATADO, index=False, encoding="utf-8")
    gerar_dicionario(PROCESSED_DIR)
    print(f"\nBase tratada: {CSV_TRATADO}")
    print(f"Dicionario de dados: {PROCESSED_DIR / 'dicionario_dados.csv'}")


def carregar_banco(df: pd.DataFrame) -> None:
    # NaN/NaT -> None para virar NULL no PostgreSQL
    carga = df.astype(object).where(df.notna(), None)
    engine = get_engine()
    with engine.begin() as conn:
        conn.execute(text(f"TRUNCATE TABLE {TABELA_TRATADA}"))
        carga.to_sql(
            TABELA_TRATADA,
            con=conn,
            if_exists="append",
            index=False,
            method="multi",
            chunksize=500,
        )
    engine.dispose()


### 3.1 Extração da camada Raw (tudo texto)

In [ ]:
df = extrair()
df.head()

### 3.2 Limpeza de texto e conversão de tipos (casting)

In [ ]:
df = limpar_texto(df)
df = converter_tipos(df)
df.dtypes

### 3.3 Valores ausentes

In [ ]:
df = tratar_nulos(df)

### 3.4 Duplicidades e regras de negócio (casos limítrofes)

In [ ]:
df = remover_duplicidades(df)
df = validar_regras(df)

### 3.5 Colunas derivadas (feature engineering)

In [ ]:
df = criar_derivadas(df)
df[["data_venda", "hora_venda", "dia_semana", "mes", "periodo_dia", "valor_total", "valor_conferido"]].head()

### 3.6 Salvamento: CSV tratado, dicionário de dados e tabela `vendas_tratadas`

In [ ]:
df = finalizar(df)
salvar_csv(df)
carregar_banco(df)
df.head()

### 3.7 Conferência no banco

In [ ]:
engine = get_engine()
with engine.connect() as conn:
    contagem = pd.read_sql(text("""
        SELECT (SELECT COUNT(*) FROM raw_vendas)       AS raw_vendas,
               (SELECT COUNT(*) FROM vendas_tratadas)  AS vendas_tratadas
    """), con=conn)
    restricoes = pd.read_sql(text("""
        SELECT conrelid::regclass AS tabela, conname AS restricao,
               CASE contype WHEN 'p' THEN 'PRIMARY KEY' ELSE 'CHECK' END AS tipo
        FROM pg_constraint
        WHERE conrelid IN ('raw_vendas'::regclass, 'vendas_tratadas'::regclass)
        ORDER BY 1, 2
    """), con=conn)
engine.dispose()
display(contagem)
display(restricoes)

## Fase 4 — Estatística descritiva e perguntas de negócio
Cada pergunta tem uma **hipótese**, a **resposta** calculada com Pandas, a **evidência** numérica, a **validação** da hipótese e um gráfico.

In [ ]:
# Fase 4: estatistica descritiva, hipoteses, respostas e graficos.
import matplotlib


import matplotlib.pyplot as plt  # noqa: E402
import pandas as pd  # noqa: E402
import seaborn as sns  # noqa: E402


COR_BASE = "#9db4c0"
COR_DESTAQUE = "#1f6f8b"


# ---------------------------------------------------------- formatacao
def brl(valor: float) -> str:
    texto = f"{valor:,.2f}".replace(",", "X").replace(".", ",")
    return "R$ " + texto.replace("X", ".")


def pct(valor: float) -> str:
    return f"{valor * 100:.1f}".replace(".", ",") + "%"


def inteiro(valor: float) -> str:
    return f"{valor:,.0f}".replace(",", ".")


def veredito(condicao: bool) -> str:
    return "Confirmada" if condicao else "Refutada"


# ------------------------------------------------------------- graficos
def _salvar(arquivo: str) -> str:
    GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(GRAFICOS_DIR / arquivo, dpi=130)
    plt.show()
    return f"graficos/{arquivo}"


def _barras(serie, titulo, xlabel, ylabel, arquivo, formato,
            horizontal=False) -> str:
    """Barras com o maior valor destacado e rotulos de dados."""
    if horizontal:
        serie = serie.sort_values(ascending=True)
    cores = [COR_DESTAQUE if v == serie.max() else COR_BASE
             for v in serie.values]
    _, ax = plt.subplots(figsize=(9, 5))
    if horizontal:
        barras = ax.barh(serie.index, serie.values, color=cores)
    else:
        barras = ax.bar(serie.index, serie.values, color=cores)
    ax.bar_label(barras, labels=[formato(v) for v in serie.values],
                 padding=3)
    ax.set_title(titulo, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.margins(x=0.12 if horizontal else 0.02, y=0.1)
    return _salvar(arquivo)


# ------------------------------------------------------------ perguntas
def q1_faturamento_filial(df) -> dict:
    fat = df.groupby("Filial")["valor_total"].sum().sort_values(
        ascending=False)
    top = fat.index[0]
    mais_vendas = df["Filial"].value_counts().idxmax()
    grafico = _barras(
        fat, "Faturamento total por filial", "Filial",
        "Faturamento (R$)", "q1_faturamento_filial.png", brl)
    return {
        "pergunta": "Qual filial apresentou o maior faturamento?",
        "hipotese": "A filial com mais vendas (transações) também é a "
                    "de maior faturamento.",
        "resposta": f"{top}, com {brl(fat.iloc[0])} "
                    f"({pct(fat.iloc[0] / fat.sum())} do total).",
        "evidencia": "; ".join(f"{f}: {brl(v)}" for f, v in fat.items()),
        "validacao": f"{veredito(top == mais_vendas)} — a filial com mais "
                     f"vendas é {mais_vendas}.",
        "grafico": grafico,
    }


def q2_qtd_vendas_filial(df) -> dict:
    qtd = df["Filial"].value_counts()
    dif = (qtd.max() - qtd.min()) / qtd.max()
    grafico = _barras(
        qtd, "Quantidade de vendas por filial", "Filial",
        "Número de vendas", "q2_qtd_vendas_filial.png", inteiro)
    return {
        "pergunta": "Qual filial realizou a maior quantidade de vendas?",
        "hipotese": "O volume de vendas é equilibrado entre as filiais "
                    "(diferença entre maior e menor inferior a 10%).",
        "resposta": f"{qtd.index[0]}, com {inteiro(qtd.iloc[0])} vendas.",
        "evidencia": "; ".join(f"{f}: {inteiro(v)}" for f, v in qtd.items()),
        "validacao": f"{veredito(dif < 0.10)} — diferença entre maior e "
                     f"menor de {pct(dif)}.",
        "grafico": grafico,
    }


def q3_faturamento_linha(df) -> dict:
    fat = df.groupby("linha_produto")["valor_total"].sum()
    fat = fat.sort_values(ascending=False)
    top = fat.index[0]
    top_itens = df.groupby("linha_produto")["Quantidade"].sum().idxmax()
    grafico = _barras(
        fat, "Faturamento por linha de produto", "Faturamento (R$)",
        "Linha de produto", "q3_faturamento_linha_produto.png", brl,
        horizontal=True)
    return {
        "pergunta": "Qual linha de produto apresentou o maior "
                    "faturamento?",
        "hipotese": "A linha líder em faturamento também é a líder em "
                    "unidades vendidas.",
        "resposta": f"{top}, com {brl(fat.iloc[0])}.",
        "evidencia": "; ".join(f"{k}: {brl(v)}" for k, v in fat.items()),
        "validacao": f"{veredito(top == top_itens)} — a líder em unidades "
                     f"é {top_itens}.",
        "grafico": grafico,
    }


def q4_avaliacao_linha(df) -> dict:
    aval = df.groupby("linha_produto")["Avaliação"].mean()
    aval = aval.sort_values(ascending=False)
    amplitude = aval.max() - aval.min()
    grafico = _barras(
        aval.round(2), "Avaliação média por linha de produto",
        "Avaliação média (0 a 10)", "Linha de produto",
        "q4_avaliacao_linha_produto.png", lambda v: f"{v:.2f}",
        horizontal=True)
    return {
        "pergunta": "Qual linha de produto recebeu a melhor avaliação "
                    "média?",
        "hipotese": "As avaliações médias são homogêneas entre as linhas "
                    "(amplitude inferior a 0,5 ponto).",
        "resposta": f"{aval.index[0]}, com média {aval.iloc[0]:.2f}.",
        "evidencia": "; ".join(f"{k}: {v:.2f}" for k, v in aval.items()),
        "validacao": f"{veredito(amplitude < 0.5)} — amplitude de "
                     f"{amplitude:.2f} ponto(s).",
        "grafico": grafico,
    }


def q5_forma_pagamento(df) -> dict:
    pag = df["forma_pagamento"].value_counts()
    share = pag / pag.sum()
    _, ax = plt.subplots(figsize=(6, 6))
    ax.pie(
        pag.values,
        labels=[f"{k} ({inteiro(v)})" for k, v in pag.items()],
        autopct=lambda p: pct(p / 100), startangle=90,
        colors=[COR_DESTAQUE, COR_BASE, "#d9e2e8"],
        explode=[0.05] + [0] * (len(pag) - 1))
    ax.set_title("Formas de pagamento", fontweight="bold")
    grafico = _salvar("q5_formas_pagamento.png")
    return {
        "pergunta": "Qual foi a forma de pagamento mais utilizada?",
        "hipotese": "Nenhuma forma de pagamento concentra mais da "
                    "metade das vendas.",
        "resposta": f"{pag.index[0]}, com {inteiro(pag.iloc[0])} vendas "
                    f"({pct(share.iloc[0])}).",
        "evidencia": "; ".join(f"{k}: {pct(s)}" for k, s in share.items()),
        "validacao": f"{veredito(share.max() <= 0.5)} — a líder detém "
                     f"{pct(share.max())}.",
        "grafico": grafico,
    }


def q6_valor_medio(df) -> dict:
    media = df["valor_total"].mean()
    mediana = df["valor_total"].median()
    _, ax = plt.subplots(figsize=(9, 5))
    sns.histplot(df["valor_total"], bins=30, kde=True,
                 color=COR_BASE, ax=ax)
    ax.axvline(media, color="crimson", linestyle="--",
               label=f"Média: {brl(media)}")
    ax.axvline(mediana, color="black", linestyle=":",
               label=f"Mediana: {brl(mediana)}")
    ax.set_title("Distribuição do valor total das vendas",
                 fontweight="bold")
    ax.set_xlabel("Valor total (R$)")
    ax.set_ylabel("Frequência")
    ax.legend()
    grafico = _salvar("q6_valor_medio.png")
    return {
        "pergunta": "Qual foi o valor médio das vendas?",
        "hipotese": "A média é maior que a mediana (distribuição "
                    "assimétrica à direita).",
        "resposta": f"{brl(media)} por venda.",
        "evidencia": f"média {brl(media)}; mediana {brl(mediana)}; "
                     f"desvio padrão {brl(df['valor_total'].std())}.",
        "validacao": f"{veredito(media > mediana)} — média "
                     f"{'>' if media > mediana else '<='} mediana.",
        "grafico": grafico,
    }


def q7_maior_venda(df) -> dict:
    linha = df.loc[df["valor_total"].idxmax()]
    q1, q3 = df["valor_total"].quantile([0.25, 0.75])
    limite = q3 + 1.5 * (q3 - q1)
    e_outlier = linha["valor_total"] > limite
    _, ax = plt.subplots(figsize=(9, 4))
    sns.boxplot(x=df["valor_total"], color=COR_BASE, ax=ax)
    ax.scatter(linha["valor_total"], 0, color="crimson", zorder=5,
               label=f"Maior venda: {brl(linha['valor_total'])}")
    ax.set_title("Posição da maior venda na distribuição",
                 fontweight="bold")
    ax.set_xlabel("Valor total (R$)")
    ax.legend()
    grafico = _salvar("q7_maior_venda.png")
    return {
        "pergunta": "Qual foi a maior venda registrada?",
        "hipotese": "A maior venda é um outlier estatístico (acima de "
                    "Q3 + 1,5 × IQR).",
        "resposta": f"{brl(linha['valor_total'])} (venda "
                    f"{linha['id_venda']}, filial {linha['Filial']}, "
                    f"linha {linha['linha_produto']}).",
        "evidencia": f"limite superior de outliers: {brl(limite)}.",
        "validacao": f"{veredito(e_outlier)} — a maior venda "
                     f"{'excede' if e_outlier else 'não excede'} "
                     f"o limite.",
        "grafico": grafico,
    }


def q8_dia_semana(df) -> dict:
    vendas = df["dia_semana"].value_counts().reindex(DIAS_SEMANA,
                                                     fill_value=0)
    top = vendas.idxmax()
    fim_semana = vendas[["Sábado", "Domingo"]].sum() / vendas.sum()
    grafico = _barras(
        vendas, "Quantidade de vendas por dia da semana",
        "Dia da semana", "Número de vendas",
        "q8_vendas_dia_semana.png", inteiro)
    return {
        "pergunta": "Em qual dia da semana ocorreu a maior quantidade "
                    "de vendas?",
        "hipotese": "O fim de semana concentra mais vendas do que o "
                    "esperado (2/7 = 28,6% do total).",
        "resposta": f"{top}, com {inteiro(vendas[top])} vendas.",
        "evidencia": "; ".join(f"{d}: {inteiro(v)}"
                               for d, v in vendas.items()),
        "validacao": f"{veredito(fim_semana > 2 / 7)} — sábado e domingo "
                     f"somam {pct(fim_semana)} das vendas.",
        "grafico": grafico,
    }


# --------------------------------------------------------------- saidas
def salvar_estatisticas(df: pd.DataFrame) -> pd.DataFrame:
    colunas = ["preco_unitario", "Quantidade", "Imposto", "valor_total",
               "custo_mercadoria", "receita_bruta", "Avaliação"]
    desc = df[colunas].describe().T.round(2)
    desc["mediana"] = df[colunas].median().round(2)
    desc.to_csv(RESULTADOS_DIR / "estatisticas_descritivas.csv",
                encoding="utf-8")
    return desc


def salvar_respostas(respostas: list[dict]) -> None:
    linhas = ["# Respostas às perguntas de negócio", ""]
    for i, r in enumerate(respostas, start=1):
        linhas += [
            f"## {i}. {r['pergunta']}",
            "",
            f"**Hipótese:** {r['hipotese']}",
            "",
            f"**Resposta:** {r['resposta']}",
            "",
            f"**Evidência:** {r['evidencia']}",
            "",
            f"**Validação da hipótese:** {r['validacao']}",
            "",
            f"![{r['pergunta']}]({r['grafico']})",
            "",
        ]
    (RESULTADOS_DIR / "respostas_negocio.md").write_text(
        "\n".join(linhas), encoding="utf-8")


In [ ]:
df = pd.read_csv(CSV_TRATADO, parse_dates=["data_venda"])
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
respostas = []


def mostrar(r):
    respostas.append(r)
    print(f"Pergunta : {r['pergunta']}")
    print(f"Hipótese : {r['hipotese']}")
    print(f"Resposta : {r['resposta']}")
    print(f"Evidência: {r['evidencia']}")
    print(f"Validação: {r['validacao']}")

### Estatística descritiva

In [ ]:
salvar_estatisticas(df)

### 1. Qual filial apresentou o maior faturamento?

In [ ]:
mostrar(q1_faturamento_filial(df))

### 2. Qual filial realizou a maior quantidade de vendas?

In [ ]:
mostrar(q2_qtd_vendas_filial(df))

### 3. Qual linha de produto apresentou o maior faturamento?

In [ ]:
mostrar(q3_faturamento_linha(df))

### 4. Qual linha de produto recebeu a melhor avaliação média?

In [ ]:
mostrar(q4_avaliacao_linha(df))

### 5. Qual foi a forma de pagamento mais utilizada?

In [ ]:
mostrar(q5_forma_pagamento(df))

### 6. Qual foi o valor médio das vendas?

In [ ]:
mostrar(q6_valor_medio(df))

### 7. Qual foi a maior venda registrada?

In [ ]:
mostrar(q7_maior_venda(df))

### 8. Em qual dia da semana ocorreu a maior quantidade de vendas?

In [ ]:
mostrar(q8_dia_semana(df))

### Salvando as respostas
Gera `resultados/respostas_negocio.md`.

In [ ]:
salvar_respostas(respostas)
print(f"Respostas salvas em {RESULTADOS_DIR / 'respostas_negocio.md'}")